In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import os
os.environ["OMP_NUM_THREADS"] = "1"
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import calinski_harabasz_score, silhouette_score
import plotly.express as px
import prince
from sklearn.decomposition import PCA

%matplotlib inline
from mpl_toolkits.mplot3d import Axes3D
plt.rcParams['figure.figsize'] = (16, 9)
plt.style.use('ggplot')

# Datos

https://www.kaggle.com/datasets/abdallahwagih/mall-customers-segmentation

In [ ]:
path = kagglehub.dataset_download("abdallahwagih/mall-customers-segmentation")

print("Path to dataset files:", path)

In [ ]:
ruta_archivo = os.path.join(path, "Mall_Customers.csv")
datos = pd.read_csv(ruta_archivo)

In [ ]:
datos.sample(5)

In [ ]:
datos.describe()

In [ ]:
print(datos.groupby('Genre').size())

In [ ]:
sb.pairplot(datos.dropna(),size=4,vars=["Age","Annual Income (k$)","Spending Score (1-100)"],kind='scatter')

# Mostramos el gráfico
plt.show()

In [ ]:
# Seleccionamos solo las variables numéricas
X = datos[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values

# Creamos la figura y el plano 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Agregamos los puntos sin escala de colores
ax.scatter(X[:, 0], X[:, 1], X[:, 2], color='blue', s=60)

# Etiquetas de los ejes
ax.set_xlabel("Edad")
ax.set_ylabel("Ingreso Anual (k$)")
ax.set_zlabel("Puntaje de Gasto (1-100)")
ax.set_title("Distribución de Clientes del Mall")

# Mostrar gráfico
plt.show()

# Segmentacion

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  

Nc = range(1, 20)

kmeans = [KMeans(n_clusters=i, random_state=42, n_init='auto') for i in Nc]

inercias = [modelo.fit(X_scaled).inertia_ for modelo in kmeans]

variaciones = [0] + [inercias[i-1] - inercias[i] for i in range(1, len(inercias))]

fig, ax1 = plt.subplots(figsize=(12, 6))

# Eje principal (Curva del Codo tradicional)
color1 = 'tab:blue'
ax1.set_xlabel('Número de Clústeres (k)', fontsize=12)
ax1.set_ylabel('Inercia (Suma de errores al cuadrado)', color=color1, fontsize=12)
ax1.plot(Nc, inercias, marker='o', color=color1, linewidth=2.5, label='Inercia')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(Nc) 
ax1.grid(True, linestyle='--', alpha=0.6)

# Segundo eje compartido (Variación / Ganancia Marginal)
ax2 = ax1.twinx()  
color2 = 'tab:red'
ax2.set_ylabel('Reducción de la Inercia (Variación)', color=color2, fontsize=12)  
# Usamos un estilo de línea distinto (punteado y marcadores cuadrados) para diferenciar
ax2.plot(Nc, variaciones, marker='s', linestyle='--', color=color2, linewidth=2, alpha=0.8, label='Reducción Marginal')
ax2.tick_params(axis='y', labelcolor=color2)

# Títulos y diseño final
plt.title('Curva del Codo y Ganancia Marginal por Clúster', fontsize=14, pad=15)
fig.tight_layout()  

# Mostrar el gráfico
plt.show()

In [ ]:
# Definir el rango de clústeres
rango_clusters = range(2, 20)

ch_scores = []
sil_scores = []

# Iterar para calcular los índices
for k in rango_clusters:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    etiquetas = kmeans.fit_predict(X_scaled) 
    ch = calinski_harabasz_score(X_scaled, etiquetas)
    sil = silhouette_score(X_scaled, etiquetas)
    ch_scores.append(ch)
    sil_scores.append(sil)

# Visualización de las métricas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico para Calinski-Harabasz
ax1.plot(rango_clusters, ch_scores, marker='s', color='green', linewidth=2)
ax1.set_title('Índice de Calinski-Harabasz\n(El pico más alto indica el k óptimo)')
ax1.set_xlabel('Número de Clústeres (k)')
ax1.set_ylabel('Puntuación CH')
ax1.set_xticks(rango_clusters)
ax1.grid(True, linestyle='--', alpha=0.7)

# Gráfico para Silhouette
ax2.plot(rango_clusters, sil_scores, marker='D', color='purple', linewidth=2)
ax2.set_title('Coeficiente de Silhouette\n(El valor más alto indica el k óptimo)')
ax2.set_xlabel('Número de Clústeres (k)')
ax2.set_ylabel('Puntuación Silhouette')
ax2.set_xticks(rango_clusters)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Imprimir los resultados exactos para análisis numérico
print("Análisis Numérico de Clústeres:")
print("-" * 40)
for i, k in enumerate(rango_clusters):
    print(f"k={k} | Calinski-Harabasz: {ch_scores[i]:.2f} | Silhouette: {sil_scores[i]:.4f}")

In [ ]:
kmeans = KMeans(n_clusters=6).fit(X_scaled)
print(kmeans.cluster_centers_)

In [ ]:
# Predecir los clusters con datos escalados
labels = kmeans.predict(X_scaled)

# Obtener los centroides y desescalarlos a la escala original
C_scaled = kmeans.cluster_centers_
C_original = scaler.inverse_transform(C_scaled)

# Asignar colores a los clusters
colores = ['red', 'green', 'blue', 'black', 'yellow', 'purple']
asignar = [colores[label] for label in labels]

# Graficar en 3D con datos originales
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=asignar, s=60)
ax.scatter(C_original[:, 0], C_original[:, 1], C_original[:, 2], marker='*', c=colores, s=1000)

# Etiquetas de los ejes
ax.set_xlabel("Edad")
ax.set_ylabel("Ingreso Anual (k$)")
ax.set_zlabel("Puntaje de Gasto (1-100)")
ax.set_title("Segmentación de Clientes")

plt.show()

In [ ]:
df_plot = datos[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()
df_plot['Cluster'] = labels.astype(str) 

# Crear el gráfico 3D interactivo
fig = px.scatter_3d(
    df_plot, 
    x='Age', 
    y='Annual Income (k$)', 
    z='Spending Score (1-100)',
    color='Cluster', 
    color_discrete_sequence=['red', 'green', 'blue', 'black', 'yellow', 'purple'], 
    title="Segmentación de Clientes Interactiva (k=6)",
    opacity=0.8, 
    size_max=10
)

# Ajustar el tamaño del gráfico
fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title="Edad",
        yaxis_title="Ingreso Anual (k$)",
        zaxis_title="Puntaje de Gasto (1-100)"
    )
)

fig.show()

In [ ]:
# Graficar en 2D
plt.figure(figsize=(8,6))
plt.scatter(X[:, 1], X[:, 2], c=asignar, s=70)
plt.scatter(C_original[:, 1], C_original[:, 2], marker='*', c=colores, s=1000)

# Etiquetas de los ejes
plt.xlabel("Ingreso Anual (k$)")
plt.ylabel("Puntaje de Gasto (1-100)")
plt.title("Segmentación de Clientes - K-Means")
plt.show()

In [ ]:
# Graficar en 2D
plt.figure(figsize=(8,6))
plt.scatter(X[:, 0], X[:, 1], c=asignar, s=70)
plt.scatter(C_original[:, 0], C_original[:, 1], marker='*', c=colores, s=1000)

# Etiquetas de los ejes
plt.xlabel("Edad")
plt.ylabel("Ingreso Anual (k$)")
plt.title("Segmentación de Clientes - K-Means")
plt.show()

In [ ]:
# Agregar el segmento al dataset original
datos['Segmento'] = labels

# Agrupar por segmento y calcular estadísticas
grouped_data = datos.groupby('Segmento').agg(
    {
        'Genre': lambda x: x.value_counts().index[0],  # Categoría más frecuente
        'Age': 'mean',
        'Annual Income (k$)': 'mean',
        'Spending Score (1-100)': 'mean',
        'Segmento': 'count'  # Cantidad de individuos por segmento
    }
).rename(columns={'Segmento': 'Cantidad'}).reset_index()

# Mostrar datos agrupados
grouped_data

# Reduccion de dimensionalidad

PCA

In [ ]:
pca_3 = PCA(n_components=3, random_state=42)
pca_3.fit(X_scaled)

# Extraer los porcentajes de varianza
varianza_individual = pca_3.explained_variance_ratio_ * 100
varianza_acumulada = np.cumsum(varianza_individual)
etiquetas = ['Componente 1', 'Componente 2', 'Componente 3']

# Construcción del Gráfico de Doble Eje
fig, ax1 = plt.subplots(figsize=(10, 6))

# Eje principal: Barras para la varianza individual
color_barras = 'steelblue'
barras = ax1.bar(etiquetas, varianza_individual, color=color_barras, alpha=0.8, width=0.5)
ax1.set_ylabel('Varianza Explicada Individual (%)', color=color_barras, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_barras)
ax1.set_ylim(0, 110) # Damos espacio extra en la parte superior

# Eje secundario: Línea para la varianza acumulada
ax2 = ax1.twinx()
color_linea = 'darkorange'
linea = ax2.plot(etiquetas, varianza_acumulada, color=color_linea, marker='o', markersize=8, linewidth=3, linestyle='--')
ax2.set_ylabel('Varianza Explicada Acumulada (%)', color=color_linea, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color_linea)
ax2.set_ylim(0, 110)

# Etiquetas para las barras
for barra in barras:
    altura = barra.get_height()
    ax1.text(barra.get_x() + barra.get_width()/2, altura + 2, 
             f'{altura:.1f}%', ha='center', va='bottom', 
             fontsize=11, color=color_barras, fontweight='bold')

# Etiquetas para la línea acumulada
for i, valor in enumerate(varianza_acumulada):
    ax2.text(i, valor - 6, 
             f'{valor:.1f}%', ha='center', va='top', 
             fontsize=11, color='darkred', fontweight='bold')

# Detalles estéticos finales
plt.title('Análisis de Varianza Explicada por Componente (PCA)', fontsize=14, pad=15)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()

plt.show()

In [ ]:
comp_x = 0  # Eje X (Ej: 0 para el Componente 1)
comp_y = 2  # Eje Y (Ej: 2 para el Componente 3)

cargas = pca_3.components_[[comp_x, comp_y], :].T

nombres_variables = ['Edad', 'Ingreso Anual (k$)', 'Puntaje de Gasto']

# Configurar la figura 
fig, ax = plt.subplots(figsize=(8, 8))

# Dibujar el círculo unitario
circulo = plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--', alpha=0.5)
ax.add_patch(circulo)

# Dibujar las flechas
colores_flechas = ['tab:red', 'tab:green', 'tab:blue']

for i, (carga_x, carga_y) in enumerate(cargas):
    ax.arrow(0, 0, carga_x, carga_y, 
             head_width=0.05, head_length=0.05, 
             fc=colores_flechas[i], ec=colores_flechas[i], alpha=0.8, linewidth=2)
    ax.text(carga_x * 1.15, carga_y * 1.15, nombres_variables[i], 
            color='black', ha='center', va='center', fontsize=11, fontweight='bold')

# Estética y ejes de referencia
plt.axhline(0, color='black', linewidth=1, alpha=0.5)
plt.axvline(0, color='black', linewidth=1, alpha=0.5)

plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)

# Etiquetas dinámicas con el porcentaje de varianza correcto
var_exp = pca_3.explained_variance_ratio_ * 100

# Sumamos 1 a los índices en el texto para que la clase vea "Componente 1" en lugar de "Componente 0"
plt.xlabel(f'Componente Principal {comp_x + 1} ({var_exp[comp_x]:.1f}%)', fontsize=12)
plt.ylabel(f'Componente Principal {comp_y + 1} ({var_exp[comp_y]:.1f}%)', fontsize=12)
plt.title(f'Círculo de Correlaciones: CP{comp_x + 1} vs CP{comp_y + 1}', fontsize=14, pad=15)
plt.grid(True, linestyle=':', alpha=0.6)

plt.show()

In [ ]:
cargas_3d = pca_3.components_.T  

nombres_variables = ['Edad', 'Ingreso Anual (k$)', 'Puntaje de Gasto']

# Configurar la figura 3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Dibujar las flechas (Vectores) desde el origen (0,0,0)
colores_flechas = ['tab:red', 'tab:green', 'tab:blue']

for i, (cx, cy, cz) in enumerate(cargas_3d):
    ax.quiver(0, 0, 0, cx, cy, cz, 
              color=colores_flechas[i], 
              arrow_length_ratio=0.15, 
              linewidth=2.5, alpha=0.8)

    ax.text(cx * 1.15, cy * 1.15, cz * 1.15, nombres_variables[i], 
            color='black', fontsize=11, fontweight='bold')

# Dibujar líneas de referencia (Ejes que cruzan por el origen)
ax.plot([-1, 1], [0, 0], [0, 0], color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.plot([0, 0], [-1, 1], [0, 0], color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.plot([0, 0], [0, 0], [-1, 1], color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Estética, límites y etiquetas de los ejes
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])

var_exp = pca_3.explained_variance_ratio_ * 100

ax.set_xlabel(f'CP 1 ({var_exp[0]:.1f}%)', fontweight='bold')
ax.set_ylabel(f'CP 2 ({var_exp[1]:.1f}%)', fontweight='bold')
ax.set_zlabel(f'CP 3 ({var_exp[2]:.1f}%)', fontweight='bold')
plt.title('Representación 3D de las Variables Originales en el Espacio PCA', fontsize=14, pad=20)

plt.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# 1. Extraer las cargas (loadings)
cargas_3d = pca_3.components_.T  
nombres_variables = ['Edad', 'Ingreso Anual (k$)', 'Puntaje de Gasto']
var_exp = pca_3.explained_variance_ratio_ * 100

# 2. Inicializar la figura interactiva
fig = go.Figure()
colores = ['red', 'green', 'blue']

# 3. Añadir cada vector como una línea desde el origen
for i, (cx, cy, cz) in enumerate(cargas_3d):
    fig.add_trace(go.Scatter3d(
        x=[0, cx], y=[0, cy], z=[0, cz],
        mode='lines+markers+text',
        line=dict(color=colores[i], width=6),
        marker=dict(size=5, color=colores[i]), 
        text=['', nombres_variables[i]], 
        textposition="top center",
        textfont=dict(size=13, color='black', family="Arial Black"),
        name=nombres_variables[i]
    ))

# ==============================================================
# CONFIGURACIÓN VISUAL DE LOS EJES (Para recuperar la cuadrícula)
# ==============================================================
eje_estilo = dict(
    range=[-1, 1],
    showbackground=True,
    backgroundcolor="whitesmoke", # Un fondo gris casi blanco para alto contraste
    showgrid=True,
    gridcolor="silver",           # Color de la cuadrícula
    zeroline=True,
    zerolinecolor="black",        # Remarca fuertemente el eje X=0, Y=0, Z=0
    zerolinewidth=3               # Grosor de la línea del origen
)

# 4. Configurar el diseño y aplicar el estilo a los 3 ejes
fig.update_layout(
    title="Representación 3D Dinámica de las Variables (Loadings PCA)",
    scene=dict(
        xaxis=dict(title=f'CP 1 ({var_exp[0]:.1f}%)', **eje_estilo),
        yaxis=dict(title=f'CP 2 ({var_exp[1]:.1f}%)', **eje_estilo),
        zaxis=dict(title=f'CP 3 ({var_exp[2]:.1f}%)', **eje_estilo),
        aspectmode='cube' 
    ),
    showlegend=True,
    margin=dict(l=0, r=0, b=0, t=50)
)

# 5. Mostrar gráfico
fig.show()

In [ ]:
from sklearn.decomposition import PCA

# 1. Inicializar PCA para 2 componentes usando la matriz numérica ya estandarizada
pca = PCA(n_components=2, random_state=42)
componentes_pca = pca.fit_transform(X_scaled)

# 2. Visualización rápida
plt.figure(figsize=(10, 6))
plt.scatter(componentes_pca[:, 0], componentes_pca[:, 1], c='gray', s=60, alpha=0.7)
plt.title("PCA (Solo Variables Numéricas)")
plt.xlabel(f"Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()